In [ ]:
!pip install torch torchvision pillow scikit-learn matplotlib

In [ ]:
from PIL import Image, ImageDraw
import os, numpy as np

CLASSES = ['normal','hairline_crack','missing_fastener','signal_failure']
IMG_PER_CLASS = 300
BASE = '/content/track_data'

for cls in CLASSES:
    os.makedirs(f'{BASE}/{cls}', exist_ok=True)

def make_image(cls, idx):
    img = Image.new('RGB', (224,224), color=(110,105,100))
    draw = ImageDraw.Draw(img)
    # Base rail lines
    draw.rectangle([80,40,100,184],  fill=(70,70,70))
    draw.rectangle([124,40,144,184], fill=(70,70,70))
    draw.rectangle([60,100,164,116], fill=(80,80,80))
    draw.rectangle([60,140,164,156], fill=(80,80,80))
    if cls == 'hairline_crack':
        x0 = np.random.randint(80,130)
        draw.line([(x0,60),(x0+20,160)], fill=(30,30,30), width=2)
    elif cls == 'missing_fastener':
        cx = np.random.randint(85,130)
        cy = np.random.randint(60,160)
        draw.ellipse([cx-8,cy-8,cx+8,cy+8], fill=(110,105,100))
    elif cls == 'signal_failure':
        draw.rectangle([30,30,80,80], fill=(180,40,40))
        draw.text((38,46),'SIG', fill=(255,255,255))
    # Add random noise
    arr = np.array(img, dtype=np.float32)
    arr += np.random.normal(0, 8, arr.shape)
    return Image.fromarray(arr.clip(0,255).astype(np.uint8))

for cls in CLASSES:
    for i in range(IMG_PER_CLASS):
        img = make_image(cls, i)
        img.save(f'{BASE}/{cls}/img_{i:04d}.jpg')

print(f"Dataset ready: {len(CLASSES)} classes x {IMG_PER_CLASS} images")

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

full = datasets.ImageFolder(BASE)
n_val = int(0.2 * len(full))
n_train = len(full) - n_val
train_set, val_set = random_split(full, [n_train, n_val])
train_set.dataset.transform = transform_train
val_set.dataset.transform   = transform_val

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)
print(f"Train: {n_train}, Val: {n_val}")

In [ ]:
import torchvision.models as models, torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
# Freeze backbone, fine-tune classifier only
for p in model.features.parameters():
    p.requires_grad = False
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 4)
)
model = model.to(device)
print("MobileNetV2 loaded, classifier replaced for 4 classes")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
best_acc = 0.0

for epoch in range(15):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'mobilenet.pth')
    print(f"Epoch {epoch+1}/15 | loss={running_loss/len(train_loader):.3f} | val_acc={val_acc:.3f}")

print(f"Best val accuracy: {best_acc:.3f}")

In [ ]:
import json
with open('classes.json','w') as f:
    json.dump(full.classes, f)
print("Exported: mobilenet.pth, classes.json")
print("Download both files \u2192 put in ml_models/ folder")